# Diffusion Models cơ bản

Diffusion Models đang dần phổ biến. Nhiều trường đại học và khóa học đã đưa Diffusion Models vào chương trình giảng dạy. Mình viết bài này với hy vọng bài viết này sẽ có ích phần nào với các bạn muốn tìm hiểu về Diffusion Models.

## Một số từ tạm dịch

- Diffusion: khuếch tán

- Quasi-static process: quá trình chuẩn tĩnh

- Thermodynamic Equilibrium: cân bằng nhiệt động học (NĐH)

- Forward Diffusion Process: qúa trình khuếch tán thuận

- Reverse Diffusion Process: quá trình đảo ngược

- Transition kernel: nhân biến đổi

## Lời mở đầu

Diffusion Models đứng sau một số sản phẩm đình đám trong lĩnh vực sinh ảnh. Ứng dụng của loại mô hình này đã được mở rộng sang Object Detection, Image Segmentation,... Danh sách này nhiều khả năng sẽ còn mở rộng hơn nữa. Bài viết này sẽ giới thiệu từ ý tưởng cơ bản, những khái niệm trong nhiệt động học đã truyền cảm hứng cho Diffusion Models trong học sâu. Sau đó, ta sẽ đi đến lý thuyết và cách huấn luyện Diffusion Models trong học sâu.

## I. Từ Nhiệt động học

$$
"\text{Ý tưởng mới thường có tính cách liên ngành}"
$$

GS Nguyễn Văn Tuấn

Artificial Neural Network (Mạng nơ-ron nhân tạo), Genetic Algorithm (Giải thuật di truyền), Attention Mechanism (Cơ chế tập trung) là những ví dụ cho câu nói trên.

Những phương pháp kể trên được lấy cảm hứng từ Khoa học thần kinh, Sinh học, Hệ thống thị giác của con người. Từ 2015, danh sách kể trên có thêm Diffusion Models.

Ý tưởng cơ bản của phương pháp này được truyền cảm hứng từ Non-equilibrium Thermodynamics (Nhiệt động học không cân bằng), một lĩnh vực đã được nghiên cứu trong một thời gian dài. Cho đến hiện tại, những phương pháp SOTA đã có nhiều thay đổi làm cho cách hoạt động ngày càng khác xa so với Duffision trong Nhiệt động học. Dù vậy, việc hiểu Diffusion trong Nhiệt động học sẽ giúp ta hình dung được cách hoạt động của Diffusion Models trong Deep Learning. Trong phần này, chúng ta sẽ tìm hiểu về Diffusion trong nhiệt động học.

### 1. Diffusion là gì?

Diffusion (Khuếch tán) là hiện tượng chuyển động của các phân tử (hoặc ion, năng lượng...) từ vùng có mật độ cao hơn sang vùng có mật độ thấp hơn.

![](image1.png)

Hình 1. minh hoạ hiện tượng khuếch tán khi ta nhỏ một giọt thuốc nhuộm vào một cốc nước. Ban đầu giọt thuốc nhuộm tập trung ở 1 vùng nhỏ trong cốc nước với mật độ cao. Trong quá trình diffusion, thuốc nhuộm dần lan sang nhiều vùng trong cốc nước và mật độ của nó cũng loãng hơn. Sau một thời gian đủ lâu, thuốc nhuộm gần như sẽ phân bố đều trong cốc nước.

### 2. Cân bằng nhiệt động học là gì?

Một hệ (system) để được gọi là Cân bằng nhiệt động học cần đồng thời đạt được các điều kiện:
1. Cân bằng về nhiệt
2. Cân bằng cơ học
3. Cần bằng hoá học
4. Cân bằng pha

![](image2.png)

Chúng ta không cần thiết phải hiểu về 4 yếu tố kể trên. Nhưng sẽ tốt hơn nếu chúng ta hình dung được những yếu tố này. Ví dụ, xét đến yếu tố đầu tiên là Cân bằng nhiệt. Một hệ được gọi là Cân bằng nhiệt nếu nhiệt độ tại mọi điểm của hệ là giống nhau. Hình 2 minh hoạ một hệ cân bằng nhiệt(bên trái) và một hệ không cân bằng nhiệt (bên phải). Ta có thể thấy ở hình bên trái nhiệt độ tại các điểm của hệ khá "giống" nhau, còn ở hình bên phải nhiệt độ ở các điểm là rất khác nhau.

### 3. Quasi-static process

![](image3.png)

![](image4.png)

![](image5.png)

Như vậy ta đã biết một hệ cân bằng NĐH phải cân bằng về nhiều mặt. Xét một hệ cân bằng NĐH sử dụng các hạt đặt trên đỉnh pít tông để nén khí như trong hình 3. Giả sử ban đầu hệ có thể tích là $V_0$ là áp suất là $P_0$. Mỗi cặp ($P_t,V_t$) sẽ xác định một trạng thái.

Giả sử các hạt này là rất nhẹ và có rất nhiều hạt trên đỉnh pít tông. Nếu ta lấy ra một hạt trên đỉnh pít tông. Thể tích khí ở bên dưới sẽ nới rộng ra một chút là đạt được trạng thái cân bằng mới $(P_1,V_1)$.

Lấy ta tiếp tục việc lấy ra dần từng hạt một cách chậm rãi ta sẽ thu được $(P_2,V_2)$, $(P_3,V_3)$, ...,$(P_T,V_T)$

Với T là số lần lấy hạt ra, và các trạng thái $(P_t,V_t)$ đều cân bằng NĐH $(1 \leq t \leq T)$.

Quá trình chuyển từ trạng thái $(P_0,V_0)$ sang $(P_T,V_T)$ cực kỳ chậm như vậy là quasi-static và được minh hoạ ở hình 4.

Lý do người ta quan tâm đến quá trình quasi-static là tất cả Quá trình có thể đảo ngược (reversible process) đều là quá trình quasi-static. Nếu ta thêm lại dần dần các hạt đã lấy ra (từng chút từng chút một như lúc lấy ra) ta sẽ thu được các trạng thái trung gian. Trong điều kiện lý tưởng (các hạt vô cùng nhẹ, không có ma sát, thả không vận tốc ban đầu,...), các trạng thái trung gian sẽ chính là đảo ngược quá trình lấy ra $(P_T,V_T), (P_{T-1},V_{T-1}),...,(P_1,V_1), (P_0,V_0)$.

Quá trình đảo ngược này được minh hoạ trong hình 5.

## II. Diffusion trong Deep Learning

Trong phần này chúng ta sẽ tìm hiểu cách ý tưởng Diffusion trong Deep Learning được triển khai.

### 1. Ý tưởng tổng quan

Ý tưởng cơ bản của Diffusion trong Deep Learning là phá hủy cấu trúc của dữ liệu một cách có hệ thống và cực kỳ chậm thông qua $\text{Quá trình khuếch tán thuận}$.

Quá trình này có tính lặp lại và được minh họa ở hình 6. Sau đó chúng ta sẽ học cách để đảo ngược quá trình này. $\text{Quá trình đảo ngược}$ được minh hoạ ở hình 7.

Cụ thể hơn, chúng ta sẽ định nghĩa quá trình khuếch tán thuận. Quá trình này chuyển phân phối phân phối dữ liệu (vốn phức tạp) sang một phân phối đơn giản và có thể dễ dàng làm việc (như lấy mẫu).

Sau đó ta sẽ học cách để đảo ngược quá trình này.

Nếu làm được điều này(thực tế là làm được), Quá trình đảo ngược sẽ được sử dụng để sinh dữ liệu. 

Trong hai quá trình này, chúng ta chỉ cần sử dụng mạng neural để học cách thực hiện quá trình đảo ngược. Quá trình thuận hoàn toàn được cố định (fully defined, fixed) trước hoặc chỉ cần học một số biến phụ. Trong bài viết này, chúng ta sẽ cố định hoàn toàn Quá trình thuận.

![](image6.png)

![](image7.png)

![](image8.png)

![](image9.png)

Hy vọng đến đây chúng ta đã hình dung được phần nào ý tưởng diffusion trong Deep Learning. Sau đây chúng ta sẽ đi vào tìm hiểu chi tiết của hai quá trình trên.

### 2. Qúa trình khuếch tán thuận

![](image10.png)

Quá trình thuận xuất phát từ phân phối của dữ liệu $q(x_0)$ và chuyển đổi dần dần thành phân phối có thể dễ dàng lấy mẫu $q(x_T) \approx N(x_t;0,I)$. Phân bố của $q(x_T)$ được chọn trước là một prior.

Trong hình 8, ở quá trình thuận, $x_0$ là dữ liệu; $x_1,x_2,...,x_T$ là các biến ẩn (latent) có cùng số chiều với $x_0$, $T$ là số bước biến đổi.

Quá trình khuếch tán được mô tả bằng một chuỗi $Markov$, nghĩa là trạng thái $x_t$ chỉ phụ thuộc vào $x_{t-1}$. Nhân biến đổi (transition kernel) của chuỗi $q(x_t|x_{t-1})$ được chọn sao cho:
$$
q(x_t|x_{t-1}) = N(x_t;x_{t-1}\sqrt{1 - \beta_t},B_tI) \space \text{với} \space B_t \in (0,1)
$$

$B_t$ là tốc độ khuyếch tán ở bước $t$. Như vậy ta có thể tính $x_t$ dựa vào $x_{t-1}$ như sau:

$$
x_t = \sqrt{1 - \beta_t}x_{t-1} + \sqrt{\beta_t}\epsilon_{t-1} \space \text{với} \space \epsilon_{t-1} \approx N(0, I)
$$

Dựa vào công thức trên ta hoàn toàn có thể xác định được $x_1$ từ $x_0$.

sau đó xác định được $x_2$ từ $x_1$.

Cứ như vậy tác sẽ tính được đến $x_T$ từ $x_{T-1}$.

Nếu ta đặt $\alpha_t := 1 - \beta_t$, $\bar{\alpha_t} := ∏^{t}_{s=1}\alpha_s$ và qua các phép biến đổi, ta có:

$$
x_t = \sqrt{\bar{\alpha_t}}x_0 + \sqrt{1 - \bar{\alpha_t}}\epsilon
$$

Đây là một tính chất quan trọng của quá trình thuận. Tính chất này trên cho phép ta lấy mẫu được $x_t$ ở một bước $t$ bất kỳ một cách trực tiếp (mà không phải đi từ bước 0, 1, 2,... rồi mới đến $t$).

Đồng thời, công thức trên cũng cho ta thấy rõ được quá trình biến đổi dần dần phân phối dữ liệu thành nhiễu đẳng hướng.

Cụ thể, vì $\beta_t < 1$ với mọi $t$, nên khi $t$ tiến tới $T$ (ví dụ $T = 1000$), $\sqrt{\bar{\alpha_t}}x_0$ sẽ tiến tới $0$ và $(1 - \bar{\alpha_t})I$ sẽ tiến tới $I$.

Phần biến đổi để thu được công thức trên các bạn có thể xem ở dưới đây:

$$
x_t = \sqrt{\alpha_t}x_{t-1} + \sqrt{1 - \alpha_t}\epsilon_{t-1} \\
= \sqrt{\alpha_t \alpha_{t-1}}x_{t-2} + \sqrt{1 - \alpha_t \alpha_{t-1}}\bar{\epsilon}_{t-2} \\
... \\
= \sqrt{\bar{\alpha}_t}x_0 - \sqrt{1 - \bar{\alpha}_t}\epsilon \\
q(x_t|x_0) = N(x_t; \sqrt{\bar{\alpha}_t}x_0, (1 - \bar{\alpha}_t)I)
$$

Phân phối của quá trình thuận thu được bằng cách bắt đầu từ $q(x_0)$ và áp dụng nhân biến đổi bên trên qua $T$ bước là:

$$
q(x_{0...T}) = q(x_0)∏^{T}_{t=1}q(x_t|x_{t-1})
$$

Như đã nói ở trên, quá trình khuếch tán thuận mà chúng ta lựa chọn đã hoàn toàn được cố định.

Như vậy chúng ta cần chọn trước các giá trị $\beta_1, \beta_2, ..., \beta_T$, còn được gọi là lịch trình phương sai. Các giá trị này cần thoả mãn hai điều kiện:

1. Tổng lượng nhiễu $\beta_1, \beta_2, ..., \beta_T$ phải đủ lớn giúp chuyển phân phối dữ liệu trở thành nhiễu đẳng hướng Gaussian.

2. Lượng nhiễu ở mỗi bước $\beta_t$ phải đủ nhỏ để có thể đảo ngược được. (Điều này tương tự như điều kiện để một quá trình là quasi-static ở phần I)

Giá trị của T phải được chọn trước. Để thoả mãn hai điều kiện trên thì $\text{T bắt buộc phải đủ lớn}$. T càng lớn thì ta có thể làm cho $\beta_t$ càng nhỏ. Để tóm tắt lại phần này mình xin trích lại slide của tác giả paper ở hình 9.

![](image11.png)

### 3. Quá trình đảo ngược

Quá trình đảo ngược cũng là một chuỗi $Markov$ có những trạng thái như quá trình thuận nhưng theo chiều ngược lại, như được thể hiện ở hình 8. Quá trình đảo ngược còn được gọi quá trình sinh. Phân phối của quá trình sinh có được qua T bước áp dụng nhân biến đổi (tương tự quá trình thuận):

$$
p(x_{0...T}) = p(x_T)∏^{T}_{t=1}p(x_{t-1}|x_t)
$$

Với $p(x_T) = N(x_T;0,I)$. Để đảo ngược được chúng ta chỉ cần xác định nhân biến đổi ngược $p(x_{t-1}|x_t)$ và biến đổi $T$ bước để thu được $x_0$.

Nếu $\beta_t$ nhỏ thì quá trình thuận và quá trình đảo ngược sẽ có cùng functional form. Vì nhân biến đổi thuận là Gaussian và $\beta_t$ đủ nhỏ, nên ta biết rằng nhân biến đổi nghịch cũng là một phân phối Gaussian.

Còn mean và covariance của phân phối này thì chúng ta có thể sử dụng mạng neural để ước lượng. Ta có thể viết nhân biến đổi ngược dưới dạng tổng quát nhất như sau:
$$
p(x_{t-1}|x_t) = N(x_{t-1};\mu_\theta (x_t,t), \sum_{\theta}(x_t,t))
$$

Chi phí tính toán của Diffusion Model chủ yếu đến từ chi phí tính toán của hai mô hình $\mu_{\theta}(x_t,t)$ và $\sum_{\theta}(x_t,t)$.

Để tóm tắt lại phần này, mình lại xin trích slide của tác giả ở hình 10.

![](image12.png)

Tips: hãy chú ý rằng các phân phối liên quan đến quá trình thuận là $q$, với quá trình đảo ngược là $p$

### 4. Hàm mục tiêu

$\text{Warning}$: Hãy sẵn sàng, đây là phần phức tạp nhất của Diffusion

#### 4.1 Tối ưu likelihood

Việc huấn luyện được thực hiện bằng cách tối ưu chặn trên của Negative Log Likelihood:
$$
L_{VLB} = \mathbb{E}_q[\log{\frac{q(x_{1:T}|x_0)}{p_{\theta(x_{0:T})}}}] \geq \mathbb{E}[-\log{p_\theta(x_0)}]
$$

Ta không thể tính toán hàm loss trên trực tiếp được vì không tính được hai phân phối thành phần trên.

Do đó, ta cần biến đổi $L_{VLB}$ như sau đưa về tính cách khoảng KL Divergence giữa các phân phối Gaussian (đó là lý do mà chúng ta phải thiết kế quá trình thuận thật cẩn thận để mọi thứ đều là Gaussian):

![](image13.png)

Hãy chú ý đến các thành phần ở dòng biến đổi cuối cùng. Ở các phần sau đây, chúng ta sẽ nói về từng thành phần này.

#### 4.2 Thành phần $L_T$

$$
L_T = D_{KL}(q(x_T|x_0) || p_{\theta}(x_T))
$$

Xét 2 thành phần của KL divergence bên trên. Thành phần đầu tiên, $q(x_T|x_0)$ chỉ phụ thuộc vào quá trình thuận và không chứa tham số tối ưu được. Thành phần thứ hai, $p_{\theta}(x_T)$ được chọn trước là $N(0,I)$. Do đó, $L_T$ là hằng số trong quá trình huấn luyện và ta có thể bỏ qua thành phần này.

#### 4.3 Thành phần $L_0$

Nếu như các transition kernel $p(x_{x-1}|x_t)$ với $t>1$ chiếu một không gian liên tục này sang không gian liên tục khác thì với $t=1$ nó cần chiếu một không gian liên tục về một không gian rời rạc (không gian của input).

Do sự khác biệt đó, thành phần cuối cùng của quá trình đảo ngược được tính theo cách riêng:

![](image14.png)

Trong đó, $D$ là chiều của dữ liệu, $i$ là chỉ số để lấy ra số chiều. Giả sử mỗi thành phần màu của ảnh đầu màu được chia vào 256 bin. Công thức trên tính xác suất $p_{\theta}(x_0|x_1)$ rơi vào bin đúng. Xác suất này tính được sử dụng CDF của phân phối Gaussian.

Công thức trên chỉ áp dụng với dữ liệu ảnh đầu vào gồm những số integer $\{ 0,1,...,255 \}$ được scale tuyến tính về $[-1,1]$.

#### 4.4 Thành phần $L_{t-1}$

$L_{t-1}$ là tổng của các KL divergence giữa $q(x_{t-1}|x_t,x_0)$ và $p_{\theta}(x_{t-1},x_t)$.  Hai thành phần này đóng vai trò tương ứng là target và prediction trong quá trình training.

Như đã nói ở trên với $\beta_t$ đủ nhỏ, $q(x_{t-1}|x_t)$ là phân bố Gaussian. Cho dù vậy, chúng ta không thể dễ dàng ước lượng được phân phối này vì nó yêu cầu sử dụng cả tập dữ liệu.

Tuy nhiên chúng ta có thể xác định được phân phối này khi đặt điều kiện trên $x_0$ bằng cách áp dụng quy tắc Bayes.

Ta có:
$$
q(x_{t-1}|x_t,x_0) = \frac{q(x_t|x_{t-1})q(x_t|x_0)}{q(x_{t-1}|x_0)}
$$

Cả ba thành phần để tính $q(x_{t-1}|x_t,x_0)$ lúc này để thuộc quá trình thuận và có thể xác định được dễ dàng. Qua các phép biến đổi ta có thể viết lại như sau:

$$
q(x_{t-1}|x_t,x_0) = N(x_{t-1};\~{\mu}_t(x_t,x_0), \~{\beta}_tI) \\
\text{với} \space \~{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t}\beta_t \\
\text{và} \space \~{\mu}_t(x_t,x_0) = \frac{1}{\sqrt{\alpha_t}}(x_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar{\alpha}_t}}\epsilon_t)
$$

Chi tiết phần biến đổi để ra được công thức trên các bạn có thể xem thêm ở đây.

Để tối ưu thành phần chính $L_{t-1}$ chúng ta cần sử dụng một mạng neural $p_\theta(x_{t-1}|x_t)$ để ước lượng $q(x_{t-1}|x_t,x_0)$.